# Section 1: Introduction

This notebook serves as the official demonstration and tutorial for the **Historical Simulation Engine**.

### Purpose
The goal is to provide a step-by-step guide for first-time users to run a full-year backtest on EURUSD using the framework's high-fidelity simulation capabilities. By the end of this tutorial, you will know how to load historical data, initialize the simulation environment, execute a backtest, and analyze the results using professional-grade metrics and forensic tools.

### Architecture
The simulation environment is designed to be **environment-agnostic**. The core trading logic (Strategy, Risk Management, Execution) remains identical to live trading. In this notebook, we 'hijack' the MetaTrader 5 API and redirect all calls to a virtual broker driven by a simulated clock and historical OHLCV data.

### Expected Runtime
- **Data Loading & Validation:** ~5 seconds
- **Simulation Execution:** ~1-2 minutes (depending on CPU and data volume)
- **Analysis & Reporting:** ~10 seconds

### Expected Outputs
1. A comprehensive **Performance Report** (Win Rate, Profit Factor, etc.).
2. **Visualizations** of the Equity Curve and Drawdown.
3. **Forensic Audit Reports** for the best and worst trades generated during the run.
4. Exported results in the `Backtest_Results/` directory.

# Section 2: Imports

We import only the necessary modules from the framework. Note that we do not redefine any business logic here; we leverage the existing production-ready code used in live trading.

In [1]:
import os
import sys
# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
if project_root not in sys.path: sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
from datetime import datetime, timedelta, timezone

# Framework Imports
from simulation.simulation_runner import SimulationRunner
from simulation.historical_data_feed import HistoricalDataFeed
from simulation.simulation_clock import SimulationClock
from simulation.simulation_broker import SimulationBroker
from simulation.simulation_account import SimulationAccount
from simulation.statistics_engine import StatisticsEngine
from simulation.simulation_environment import env
from trade_auditor import TradeAuditor
from Collecting_Data.position_lifecycle import PositionLifecycle

# Suppress verbose logging for the tutorial
logging.basicConfig(level=logging.ERROR)

# Section 3: Configuration

Define the parameters for our backtest. These are the primary 'knobs' a user can turn to test different symbols, timeframes, or historical periods.

In [2]:
SYMBOL = "EURUSD_o"
PRIMARY_TIMEFRAME = "M5"
SECONDARY_TIMEFRAME = "M15"
LOOKBACK_DAYS = 365
INITIAL_BALANCE = 5000
DATA_DIRECTORY = os.path.join(project_root, "Data")
RESULT_DIRECTORY = os.path.join(project_root, "Backtest_Results")

# Section 4: Determine Date Range

We dynamically calculate the start and end dates based on the current system time to ensure the notebook always tests the most recent year of available data.

In [3]:
today = datetime.now(timezone.utc)
start_date = today - timedelta(days=LOOKBACK_DAYS)

print(f"Requested Date Range: {start_date.date()} to {today.date()}")

Requested Date Range: 2025-07-14 to 2026-07-14


# Section 4.5: Synthetic Data Generation

If the historical data files are not present in the `Data/` directory, we generate synthetic data to allow the tutorial to run autonomously.

In [4]:
def generate_synthetic_data(symbol, timeframe_minutes, days):
    filename = os.path.join(DATA_DIRECTORY, f"{symbol}_M{timeframe_minutes}.csv")
    if os.path.exists(filename):
        print(f"Data file already exists: {filename}")
        return
    
    print(f"Generating synthetic {symbol} M{timeframe_minutes} data...")
    os.makedirs(DATA_DIRECTORY, exist_ok=True)
    
    end_time = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    start_time = end_time - timedelta(days=days)
    freq = f"{timeframe_minutes}min"
    date_range = pd.date_range(start=start_time, end=end_time, freq=freq, tz=timezone.utc)
    n = len(date_range)
    
    np.random.seed(42 if timeframe_minutes == 5 else 43)
    t = np.linspace(0, 1, n)
    price = 1.1000 + 0.05 * t + np.random.normal(0, 0.0001, n)
    
    df = pd.DataFrame({
        'Datetime': date_range,
        'Open': price,
        'High': price + np.random.uniform(0, 0.0005, n),
        'Low': price - np.random.uniform(0, 0.0005, n),
        'Close': price + np.random.normal(0, 0.0001, n),
        'TickVolume': np.random.randint(100, 1000, n),
        'Spread': [1] * n
    })
    df['High'] = df[['Open', 'High', 'Low', 'Close']].max(axis=1)
    df['Low'] = df[['Open', 'High', 'Low', 'Close']].min(axis=1)
    
    df.to_csv(filename, index=False)
    print(f"Saved to {filename}")

generate_synthetic_data(SYMBOL, 5, LOOKBACK_DAYS)
generate_synthetic_data(SYMBOL, 15, LOOKBACK_DAYS)

Data file already exists: c:\Users\MHossein\Documents\GitHub\Forex_DNN\Data\EURUSD_o_M5.csv
Data file already exists: c:\Users\MHossein\Documents\GitHub\Forex_DNN\Data\EURUSD_o_M15.csv


# Section 5: Load Historical Data

Data quality is paramount. We load our M5 and M15 CSV files and perform a series of sanity checks to ensure the simulation is built on a solid foundation.

In [ ]:
def verify_data(df, name, requested_start):
    print(f"Verifying {name}...")
    print(f" - Row count: {len(df)}")
    print(f" - NaNs: {df.isnull().sum().sum()}")
    print(f" - Duplicates: {df['Datetime'].duplicated().sum()}")
    
    data_start = df['Datetime'].min()
    data_end = df['Datetime'].max()
    print(f" - Data Range: {data_start} to {data_end}")
    
    tf_mins = 5 if "M5" in name else 15
    expected_bars = (data_end - data_start).total_seconds() / (tf_mins * 60) + 1
    missing = expected_bars - len(df)
    print(f" - Missing candles (including weekends): {int(missing)}")
    print("-----------------------------------")

m5_path = os.path.join(DATA_DIRECTORY, f"{SYMBOL}_{PRIMARY_TIMEFRAME}.csv")
m15_path = os.path.join(DATA_DIRECTORY, f"{SYMBOL}_{SECONDARY_TIMEFRAME}.csv")

df_m5 = pd.read_csv(m5_path, parse_dates=['Datetime'])
df_m15 = pd.read_csv(m15_path, parse_dates=['Datetime'])

verify_data(df_m5, "M5 Data", start_date)
verify_data(df_m15, "M15 Data", start_date)

# Section 6: Initialize Simulation

Now we assemble the simulation components using the framework interfaces. We explicitly create the data feed, clock, account, and broker to understand the architectural orchestration.

In [ ]:
data_files = {
    (SYMBOL, PRIMARY_TIMEFRAME): m5_path,
    (SYMBOL, SECONDARY_TIMEFRAME): m15_path
}

# 1. Create Historical Data Feed
data_feed = HistoricalDataFeed()
for (s, tf), path in data_files.items():
    data_feed.load_csv(s, tf, path)

# 2. Initialize Simulation Clock at the start of data
start_time = data_feed.get_current_bar(SYMBOL, PRIMARY_TIMEFRAME)['Datetime']
clock = SimulationClock(start_time)

# 3. Create Simulation Account
account = SimulationAccount(initial_balance=INITIAL_BALANCE, leverage=100)

# 4. Create Simulation Broker
broker = SimulationBroker(account, clock)
broker.set_symbol_info(SYMBOL, {
    "digits": 5, "point": 0.00001, "volume_min": 0.01, "volume_step": 0.01,
    "volume_max": 100.0, "trade_contract_size": 100000, "trade_stops_level": 0
})

# 5. Assemble into SimulationRunner
runner = SimulationRunner(
    symbol=SYMBOL,
    timeframes=[PRIMARY_TIMEFRAME, SECONDARY_TIMEFRAME],
    data_files=data_files,
    initial_balance=INITIAL_BALANCE,
    journal_root=RESULT_DIRECTORY
)

print("Simulation environment initialized successfully.")

# Section 7: Run Backtest

We execute the simulation loop. The engine will step through every bar, updating market prices and allowing the strategy to make decisions.

In [ ]:
import time
start_perf = time.perf_counter()

print(f"Starting simulation for {SYMBOL}...")
runner.run()
end_perf = time.perf_counter()

simulation_duration = end_perf - start_perf
print(f"\nProcessed {len(df_m5)} candles.")
print(f"Simulation completed in {simulation_duration:.2f} seconds.")

# Section 8: Performance Summary

After the simulation finishes, we reconstruct all trades and calculate aggregate metrics. This provides a high-level view of the strategy's profitability and risk profile.

In [ ]:
auditor = TradeAuditor(journal_root=RESULT_DIRECTORY, mode="backtest")
lifecycles = auditor.reconstruct_all()

stats = StatisticsEngine(lifecycles)
metrics = stats.calculate_metrics()

if metrics:
    print("--- PERFORMANCE SUMMARY ---")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k:<25}: {v:>10.2f}")
        else:
            print(f"{k:<25}: {v:>10}")
else:
    print("No trades were executed during the simulation period.")

# Section 9: Equity Curve

A visual representation of how the account balance and equity evolved over time. We also plot the drawdown to visualize the risk taken.

In [ ]:
if lifecycles:
    df_stats = stats.df.sort_values('outcome_exit_timestamp')
    df_stats['cum_profit'] = df_stats['outcome_realized_profit'].cumsum()
    df_stats['balance'] = INITIAL_BALANCE + df_stats['cum_profit']
    
    # Calculate Drawdown
    df_stats['peak'] = df_stats['balance'].cummax()
    df_stats['drawdown'] = (df_stats['balance'] - df_stats['peak'])
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    ax1.plot(df_stats['outcome_exit_timestamp'], df_stats['balance'], label='Balance')
    ax1.set_title(f'Equity Curve - {SYMBOL}')
    ax1.set_ylabel('Balance ($)')
    ax1.legend()
    ax1.grid(True)
    
    ax2.fill_between(df_stats['outcome_exit_timestamp'], df_stats['drawdown'], 0, color='red', alpha=0.3, label='Drawdown')
    ax2.set_title('Account Drawdown')
    ax2.set_ylabel('Drawdown ($)')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data to plot equity curve.")

# Section 10: Position Analysis

Here we dig into the details of individual trades to identify patterns in winners and losers.

In [ ]:
if not stats.df.empty:
    print("Top 10 Winners:")
    display(stats.df.nlargest(10, 'outcome_realized_profit')[['execution_ticket', 'outcome_realized_profit', 'outcome_duration']])
    
    print("\nTop 10 Losers:")
    display(stats.df.nsmallest(10, 'outcome_realized_profit')[['execution_ticket', 'outcome_realized_profit', 'outcome_duration']])
    
    longest_trade_idx = stats.df['outcome_duration'].idxmax()
    longest_trade = stats.df.loc[longest_trade_idx]
    print(f"\nLongest Trade: Ticket {longest_trade['execution_ticket']} (Duration: {longest_trade['outcome_duration']:.0f}s)")
    
    if 'management_maximum_adverse_excursion' in stats.df.columns:
        largest_mae_idx = stats.df['management_maximum_adverse_excursion'].idxmax()
        largest_mae_trade = stats.df.loc[largest_mae_idx]
        print(f"Largest Drawdown Trade (MAE): Ticket {largest_mae_trade['execution_ticket']} (MAE: {largest_mae_trade['management_maximum_adverse_excursion']:.2f})")
    
    if 'outcome_r_multiple' in stats.df.columns:
        print(f"Best R Multiple: {stats.df['outcome_r_multiple'].max():.2f}")
        print(f"Worst R Multiple: {stats.df['outcome_r_multiple'].min():.2f}")

# Section 11: TradeAuditor Example

The `TradeAuditor` is a forensic tool that reconstructs the complete lifecycle of a single trade. It is invaluable for debugging why a specific trade succeeded or failed.

In [ ]:
if not stats.df.empty:
    best_ticket = int(stats.df.nlargest(1, 'outcome_realized_profit').iloc[0]['execution_ticket'])
    worst_ticket = int(stats.df.nsmallest(1, 'outcome_realized_profit').iloc[0]['execution_ticket'])
    
    print(f"--- Forensic Audit: Best Trade (Ticket {best_ticket}) ---")
    lc_best = auditor.reconstruct_trade_lifecycle(ticket=best_ticket)
    if lc_best: print(auditor.format_report_markdown(lc_best))
    
    print(f"\n--- Forensic Audit: Worst Trade (Ticket {worst_ticket}) ---")
    lc_worst = auditor.reconstruct_trade_lifecycle(ticket=worst_ticket)
    if lc_worst: print(auditor.format_report_markdown(lc_worst))

# Section 12: Journal Inspection

We inspect the raw `PositionLifecycle` records saved in the journal. These records serve as the permanent, immutable history of the strategy's activity.

In [ ]:
if not stats.df.empty:
    print("Journal Record Samples:")
    display(stats.df.head(2))
    display(stats.df.tail(2))
    
    print("\nAvailable Columns:")
    print(stats.df.columns.tolist())
    
    print("\nEvent Counts:")
    journal_df = auditor.load_journal_data()
    if not journal_df.empty:
        print(journal_df['event_type'].value_counts())

# Section 13: Export

Finally, we save all generated reports and statistics to the result directory for future reference.

In [ ]:
if not stats.df.empty:
    os.makedirs(RESULT_DIRECTORY, exist_ok=True)
    stats.df.to_csv(os.path.join(RESULT_DIRECTORY, "full_trade_history.csv"), index=False)
    
    with open(os.path.join(RESULT_DIRECTORY, "metrics_summary.json"), "w") as f:
        import json
        json.dump(metrics, f, indent=4)
        
    print(f"Results exported to {RESULT_DIRECTORY}/")

# Section 14: Conclusions

The simulation run is complete. Below we summarize the execution and provide next steps for further analysis.

In [ ]:
if 'lifecycles' in locals():
    print("Simulation completed successfully")
    print(f"Execution time: {simulation_duration:.2f}s")
    print(f"Total trades: {len(lifecycles)}")
    print(f"Location of reports: {RESULT_DIRECTORY}/")
    
    print("\nSuggested next analyses:")
    print("1. Parameter Optimization: Try changing the EMA periods in MMStrategy to see how they impact the Win Rate.")
    print("2. Different Symbols: Run this notebook again for GBPUSD_o (after downloading its data).")
    print("3. Walk-Forward Analysis: Segment the historical data into 'In-Sample' for training and 'Out-of-Sample' for validation.")